In [11]:
!uv pip install -q --reinstall "numpy==1.26.4"
!uv pip install -q torch torchvision torchaudio
!uv pip install -q gradio openai-whisper ffmpeg-python
!uv pip install -q ollama

# Install ffmpeg via homebrew if not already installed (Mac)
!which ffmpeg || brew install ffmpeg

/usr/local/bin/ffmpeg


In [12]:
import os
import numpy as np
import gradio as gr
import whisper
import torch
import ollama

In [13]:
# Load Whisper model (using 'base' for faster performance on both GPU and CPU)
print("Loading Whisper model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

whisper_model = whisper.load_model("base", device=device)
print("Model loaded successfully!")
print(f"Model type: {type(whisper_model)}")
print(f"Has transcribe method: {hasattr(whisper_model, 'transcribe')}")


Loading Whisper model...
Using device: cpu
Model loaded successfully!
Model type: <class 'whisper.model.Whisper'>
Has transcribe method: True


In [14]:
# Model loaded and ready to use!


In [15]:
def transcribe_audio(audio_file, target_language):
    """Transcribe audio file to text in the specified language."""
    if audio_file is None:
        return "Please upload an audio file."
    
    try:
        # Language codes for Whisper
        language_map = {
            "English": "en",
            "Spanish": "es",
            "French": "fr",
            "German": "de",
            "Italian": "it",
            "Portuguese": "pt",
            "Chinese": "zh",
            "Japanese": "ja",
            "Korean": "ko",
            "Russian": "ru",
            "Arabic": "ar",
            "Auto-detect": None
        }
        
        lang_code = language_map.get(target_language)
        
        # Get file path from Gradio File component (returns path string directly)
        audio_path = audio_file.name if hasattr(audio_file, 'name') else audio_file
        
        if not audio_path or not os.path.exists(audio_path):
            return "Invalid audio file or file not found"

        # Transcribe using whisper_model.transcribe()
        result = whisper_model.transcribe(
            audio_path,
            language=lang_code,
            task="transcribe"
        )
        
        return result["text"]
    
    except Exception as e:
        return f"Error: {str(e)}"


In [16]:
def process_with_llm(transcription, llm_model, custom_prompt):
    """Process transcription with selected LLM."""
    if not transcription or transcription.startswith("Error") or transcription.startswith("Please"):
        return "Please transcribe audio first."
    
    try:
        # Default prompt if none provided
        if not custom_prompt:
            custom_prompt = "Summarize the following transcription:"
        
        # Construct the full prompt
        full_prompt = f"{custom_prompt}\n\n{transcription}"
        
        # Call Ollama
        response = ollama.chat(
            model=llm_model,
            messages=[
                {"role": "user", "content": full_prompt}
            ]
        )
        
        return response['message']['content']
    
    except Exception as e:
        return f"Error processing with LLM: {str(e)}\n\nNote: Make sure Ollama is running locally with the selected model installed."


In [17]:
def process_audio_pipeline(audio_file, target_language, llm_model, custom_prompt):
    """Complete pipeline: transcribe then process with LLM."""
    # Step 1: Transcribe
    transcription = transcribe_audio(audio_file, target_language)
    
    # Step 2: Process with LLM
    llm_output = process_with_llm(transcription, llm_model, custom_prompt)
    
    return transcription, llm_output


In [18]:
# Create simple Gradio interface using Interface API (more stable)
app = gr.Interface(
    fn=transcribe_audio,
    inputs=[
        gr.File(label="Upload Audio File", file_types=["audio"]),
        gr.Dropdown(
            choices=[
                "English", "Spanish", "French", "German", "Italian",
                "Portuguese", "Chinese", "Japanese", "Korean",
                "Russian", "Arabic", "Auto-detect"
            ],
            value="English",
            label="Language"
        )
    ],
    outputs=gr.Textbox(label="Transcription", lines=15),
    title="🎙️ Audio File Transcription",
    description="Upload an audio file to transcribe it.",
    allow_flagging="never"
)

print("App ready! Run the next cell to launch.")


App ready! Run the next cell to launch.


/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


In [19]:
# Launch the app
app.launch(inbrowser=True)  # inbrowser=True opens browser automatically


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/protocols/http/httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/fastapi/applications.py", line 1133, in __call__
    await super().__call__(scope, receive, send)
  File "/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/Us

## 📋 Setup Instructions

### For Mac (Local):
1. Make sure Ollama is installed and running:
   ```bash
   ollama serve
   ```
2. Pull at least one model:
   ```bash
   ollama pull llama3.2
   ```
3. Run all cells above

### For Google Colab:
1. Install Ollama in Colab (run this in a cell):
   ```python
   !curl -fsSL https://ollama.com/install.sh | sh
   !nohup ollama serve &
   !sleep 5 && ollama pull llama3.2:1b
   ```
2. Run all cells above

### Usage:
1. Upload an MP3/audio file
2. Select your preferred language (default: English)
3. Choose an LLM model
4. (Optional) Add a custom prompt for the LLM
5. Click "Process Audio"
6. View transcription and LLM analysis!


In [20]:
# Optional: Check available Ollama models
try:
    models_list = ollama.list()
    print("✅ Ollama is running!")
    print("\n📦 Available models:")
    for m in models_list['models']:
        print(f"  - {m['model']}")
except Exception as e:
    print("❌ Ollama is not running or not accessible")
    print(f"Error: {e}")
    print("\nTo start Ollama, run: ollama serve")


✅ Ollama is running!

📦 Available models:
  - llama3.2:latest
  - gpt-oss:20b-cloud
  - gpt-oss:120b-cloud
  - deepseek-v3.1:671b-cloud
  - gemma3:270m
  - deepseek-r1:1.5b


/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/Users/hopeogbons/Projects/andela/llm_engineering/.venv/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
